## Notebook09

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
movie = (
    pl.read_csv(ub + "data/movies_50_years.csv")
    .select(
        c.year,
        c.title,
        c.mpa,
        c.runtime,
        c.gross,
        c.rating,
        c.rating_count,
        c.metacritic
    )
)

### Questions

This week's data is the hundred highest grossing films in the United States in
each year from 1970 to 2019, which makes 5,000 rows. Each row is one film. The
`gross` column is domestic box office in millions of dollars, not adjusted for
inflation, so the numbers rise over the fifty years for reasons that have nothing
to do with how many tickets were sold. The `rating` is the IMDb user score out of
ten and `rating_count` is how many people voted; `metacritic` is a critic score
out of a hundred; `mpa` is the rating label (G, PG, R, and so on) and `runtime`
is in minutes.

Last time the table was complete and every group had something in it. This one is
full of holes, which is the point: this notebook is mostly about counting, and the
interesting thing about a count is what it does not include. Several questions
below group by decade, which is not a column in the data. Build it with integer
division, as in `(c.year // 10) * 10`.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Add a decade column and count the films in each decade with `pl.len()`.

A thousand films in each decade, which is what a hundred films a year for ten
years should give. Nothing is missing.

2. Now count the same groups a different way. Alongside `pl.len()`, use `.count()`
on the `metacritic` column and on `rating`. Then say what the three numbers are
counting.

**Answer**:
every time. `.count()` counts the values that are actually there, skipping the
nulls, so it is smaller wherever the column has holes. IMDb ratings are nearly
complete after 1980 but missing for a third of the 1970s films. Metacritic is
worse and in a much more systematic way: 73 scores in the 1970s against 445 in the
2010s. Metacritic did not launch until 2001. Everything before it was scored
later, if anyone got around to it, and mostly nobody did.

3. That gap changes how you read an average. Compute the mean Metacritic score by
decade with the count sitting next to it.

Every decade lands within about two points of 50, which looks like a tidy finding:
critics have been consistent for fifty years. Before believing it, check whether
the 73 films that carry a 1970s score resemble the 927 that do not. Group the
1970s films by whether the score is missing and compare them.

**Answer**:
and $31 million at the box office; the unscored ones average 6.1 and $17 million.
Whoever went back and rated old films rated the ones people still watch. So the
1970s mean of 51.8 is the average score of the decade's memorable films, and the
2010s mean of 50.5 is the average score of nearly half of everything released.
Putting them in the same column of the same table invites a comparison that the
data cannot support. This is why `.count()` belongs next to any group mean you
intend to trust.

4. `.n_unique()` counts distinct values rather than rows, which is how you find
out whether a column identifies a row. Count the films in each decade and the
distinct titles in each decade side by side.

The 1970s have 1,000 films and 997 titles, so three titles appear twice. Titles
are not unique, and a film is identified by its title and year together, not by
its title. Across the whole table it happens 98 times: there are three films
called `Halloween` here, from 1978, 2007, and 2018.

5. `.agg()` belongs to a grouping, and there is no group here, so the obvious way
to summarize the whole table does not work. Try it and read the error.

**Answer**:
no such method, because `.agg()` only exists on the `GroupBy` object that
`.group_by()` hands back. Summarize a whole frame with `.select()` instead, which
you already know as a way to pick columns. Give it aggregating expressions and it
collapses everything to a single row. Do that now: count the rows, count the
distinct titles, count the films with a Metacritic score, and take the mean rating
and the total gross.

One row, five columns. Fifty years of American cinema, or at least the profitable
end of it, comes to $264 billion.

6. The summary functions from the end of the chapter work the same way inside
`.agg()`. For each decade compute the smallest gross, the median, the ninetieth
percentile with `.quantile(0.9)`, and the largest.

Read the median column down and the box office grows by a factor of twelve, from
$5 million to $60 million, most of which is inflation and ticket prices rather
than audiences. Then read the minimum column, which is where the data stops making
sense. The smallest gross in the 1970s is 0.000005, which is five dollars. A film
that took five dollars is not among the hundred highest grossing films of its year,
so that number is not a small gross; it is a missing gross wearing a disguise. The
minimum and the maximum are the cheapest data check you own, and they are worth
running on any column before you trust anything else about it.

7. `pl.corr()` takes two columns instead of one and measures how tightly they move
together, on a scale from -1 to 1. Do critics and audiences agree? Correlate
`rating` with `metacritic` for the whole table, and then again within each decade.

About 0.36, and remarkably steady from decade to decade. Critics and audiences
point in the same direction and not much more than that; knowing that reviewers
liked a film tells you rather little about whether people did. Remember what this
number is computed from: only the 1,536 films that have both scores, which by
question 2 is a particular kind of film.

8. Now a question that aggregation cannot answer. Which film was the highest
grossing of each decade? Compute the largest gross per decade.

**Answer**:
it takes the thousand rows of a group and returns one row, and the titles went into
that reduction and did not come out. This is what every `.agg()` does, and it is
the one thing it cannot be talked out of. Chapter 6 is about getting a group-level
number without paying this price.

9. Meanwhile there is a way to keep a title, though it takes some care. Sort the
films by gross, largest first, and use `.first()` inside `.agg()` to grab the title
at the top of each decade. Keep `.max()` next to it. Look hard at the answer before
you accept it.

**Answer**:
columns disagree with each other, which is the tell: `.max()` skips nulls and
`.sort()` does not. There are 495 films with no recorded gross at all, Polars sorts
nulls to the *top* of a descending sort, and so `.first()` handed us a film with no
box office rather than the largest one. (The 1970s answer, a 1971 film called
`Grease` with a 4.6 rating, is not the musical. It is a different film that happens
to share the name, which is question 4 coming back to bite.) The lesson is that
`.max()` and `.first()` count missing values differently, exactly as `pl.len()` and
`.count()` do.

10. Fix it. Drop the films with no recorded gross before sorting, and run the same
chain again.

Star Wars, E.T., Titanic, Avatar, and The Force Awakens. Now the two columns agree,
and so does everything you already knew about movies before you opened the file.
That last part is not a small thing. The reason we caught the bug in question 9 is
that the answer was famous enough to be obviously wrong, and most wrong answers are
not so considerate.